# MTH 4300 Activity Day 20
## Operator Overloading: Friend Functions, Default Arguments

*Reminder: don't memorize signatures. For each choice, ask yourself what the operator does with built-in types (`int`, `double`) — the answer tells you how to write it.*

This notebook has runnable starter code for each question. Use it to test your answers after you write them on paper.

## Question 1 — `Fraction == int`

We want `2 == f` to work where `f` is a `Fraction`. Figure out when a fraction equals an int, find the bug in the student's division attempt, then write a correct version.

In [1]:
////fractionEqInt.cpp
#include<iostream>

class Fraction {
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}
    int getNum() const { return this->num_; }
    int getDen() const { return this->den_; }
    friend bool operator==(int n, const Fraction& other);

};

// Write your operator== here
bool operator==(int n, const Fraction& other){
    //return other.num_ == n * other.den_;
    return other.num_ / other.den_ == n && other.num_ % other.den_ == 0;
}

int main(){
    Fraction f(6, 3);
    if (2 == f) { std::cout << "equal!\n"; }

    Fraction g(7, 3);
    if (2 == g) { std::cout << "g equals 2\n"; }
    else        { std::cout << "g does not equal 2\n"; }
}

wrote file fractionEqInt.cpp
$> g++   fractionEqInt.cpp    -o fractionEqInt
$> ./fractionEqInt 
equal!
g does not equal 2


## Question 2 — `operator+` using `operator+=`

Add `operator+` to the `Compl` class, using `operator+=` internally instead of touching `re_` and `im_` directly. (Hint: make a copy, modify the copy, return it.)

In [1]:
////complPlusFromPlusEquals.cpp
#include<iostream>

class Compl {
private:
    double re_;
    double im_;
public:
    Compl(double r, double i) : re_(r), im_(i) {}
    double getRe() const { return this->re_; }
    double getIm() const { return this->im_; }

    void display() const {
        if (this->im_ >= 0)
            std::cout << this->re_ << " + " << this->im_ << "i";
        else
            std::cout << this->re_ << " - " << -this->im_ << "i";
    }

    Compl& operator+=(const Compl& other) {
        this->re_ += other.re_;
        this->im_ += other.im_;
        return *this;
    }

    // Write operator+ here, using operator+= internally.
    Compl operator+(const Compl& other) const{
        Compl result(*this);
        result += other;
        return result;
    }

};

int main(){
    Compl a(1, 2);
    Compl b(3, 4);
    Compl c = a + b;
    c.display();   // expect: 4 + 6i
    std::cout << "\n";
}

wrote file complPlusFromPlusEquals.cpp
$> g++   complPlusFromPlusEquals.cpp    -o complPlusFromPlusEquals
$> ./complPlusFromPlusEquals 
4 + 6i


## Question 3 — Buggy `operator-=`

This code compiles. Trace through it carefully: what is `a` before the `-=`, and what is `a` after? Is that what `a -= b` should do?

In [ ]:
////complBuggyMinusEquals.cpp
#include<iostream>

class Compl {
private:
    double re_;
    double im_;
public:
    Compl(double r, double i) : re_(r), im_(i) {}

    void display() const {
        if (this->im_ >= 0)
            std::cout << this->re_ << " + " << this->im_ << "i";
        else
            std::cout << this->re_ << " - " << -this->im_ << "i";
    }

    Compl operator-=(const Compl& other) const {
        return Compl(this->re_ - other.re_, this->im_ - other.im_);
    }
};

int main(){
    Compl a(5, 2);
    Compl b(1, 1);
    a -= b;
    a.display();
    std::cout << "\n";
}

## Question 4 — `operator<<` for `Compl`

Write `operator<<` as a friend so `std::cout << z` prints `a + bi` / `a - bi`.

In [ ]:
////complStream.cpp
#include<iostream>

class Compl {
private:
    double re_;
    double im_;
public:
    Compl(double r, double i) : re_(r), im_(i) {}
    double getRe() const { return this->re_; }
    double getIm() const { return this->im_; }

    // Write friend operator<< here

};

int main(){
    Compl z(3, 4);
    Compl w(2, -5);
    std::cout << z << "\n";   // expect: 3 + 4i
    std::cout << w << "\n";   // expect: 2 - 5i
}

## Question 5 — `operator>>` for `Compl`

Write `operator>>` as a friend so `std::cin >> z` reads two doubles (real and imaginary) separated by whitespace.

In [ ]:
////complInput.cpp
#include<iostream>
#include<sstream>

class Compl {
private:
    double re_;
    double im_;
public:
    Compl(double r, double i) : re_(r), im_(i) {}
    double getRe() const { return this->re_; }
    double getIm() const { return this->im_; }

    // Write friend operator<< here (needed to print)
friend std::ostream& operator<<(std::ostream& out, const Compl& z);
    // Write friend operator>> here
friend std::instream& operator>>(std::stream& in, Compl& z);
};

std::ostream& operator<<(std::ostream& os, const Compl& z){

    os << z.re_;
    if(z.im_ >= 0){
        os << "+" << z.im_ << "i";
    }
    else{
        os << " - " << (-z.im_) << "i";
    }
    return os;
}
std::instream& operator >>(std::stream& in, Compl& z){
in >> z.re_ >> z.im_;
return in;
}


int main(){
    // Simulate input with a stringstream so the cell can run without user input
    std::istringstream input("3 4");
    Compl z(0, 0);
    input >> z;
    std::cout << z << "\n";   // expect: 3 + 4i
}

## Question 6 — Friend-vs-member theory

Conceptual. No running code needed — but you can try the experiments described in (b) and (c) below by editing this cell.

In [ ]:
////fractionFriendTheory.cpp
#include<iostream>

class Fraction {
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}
    int getNum() const { return this->num_; }
    int getDen() const { return this->den_; }

    friend Fraction operator*(const Fraction& a, const Fraction& b) {
        return Fraction(a.num_ * b.num_, a.den_ * b.den_);
    }
};

int main(){
    Fraction a(1, 2), b(3, 5);
    Fraction c = a * b;
    std::cout << c.getNum() << "/" << c.getDen() << "\n";   // expect: 3/10
}

## Question 7 — Default arguments

Predict what each `Compl` looks like, and figure out which default-argument patterns are legal.

In [ ]:
////greetDefault.cpp
// The `greet` example from the activity handout — run it to see what
// default arguments look like in action.
#include<iostream>
#include<string>

void greet(std::string name = "friend") {
    std::cout << "Hello, " << name << "!\n";
}

int main(){
    greet();           // prints: Hello, friend!
    greet("Areeba");   // prints: Hello, Areeba!
}

In [ ]:
////complDefaults.cpp
#include<iostream>

class Compl {
private:
    double re_;
    double im_;
public:
    Compl(double r = 0, double i = 0) : re_(r), im_(i) {}
    double getRe() const { return this->re_; }
    double getIm() const { return this->im_; }
};

int main(){
    Compl a;
    Compl b(3);
    Compl c(3, 4);
    std::cout << "a: " << a.getRe() << ", " << a.getIm() << "\n";
    std::cout << "b: " << b.getRe() << ", " << b.getIm() << "\n";
    std::cout << "c: " << c.getRe() << ", " << c.getIm() << "\n";
}

## Question 8 — Put it all together

Build your own complete `Compl` class. Do as many of the following as you can — start from the top and work down:

- Constructor with default arguments so that `Compl()`, `Compl(3)`, and `Compl(3, 4)` all work
- Getters `getRe()` and `getIm()`
- `operator+`, `operator-`, `operator*` &nbsp; (recall: $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$)
- `operator+=`, `operator-=`
- `operator==` (two `Compl`s are equal when both parts match)
- `operator<<` for printing in the form `a + bi` / `a - bi`
- `operator>>` for reading from input
- `operator*=`
- `operator==` that lets you compare a `Compl` to a `double` (on either side)
- Any other operator you think would be useful

In [ ]:
////complComplete.cpp
#include<iostream>

class Compl {
private:
    double re_;
    double im_;
public:
    // your code here
};

int main(){
    // test your class here
}

## Bonus — Two classes sharing a friend function

So far, every friend function we've seen has been a friend of *one* class. But a single free function can be declared a `friend` of **two different classes**, and then it has access to the privates of both. This is how you write operations that meaningfully involve two different types.

Imagine a `Thermostat` (knows the target and current temperature) and a `Heater` (knows whether it's currently on). We want a function `sync()` that reads the thermostat and flips the heater on or off to match.

- **(a)** The declaration `friend void sync(...)` appears in **both** classes. Why is that necessary? What would happen if it only appeared in one?
- **(b)** `sync` is not a member of either class, and has no `this`. Yet it reaches into the privates of both. What does this tell you about what `friend` really is — a property of the function, or a property of the class?
- **(c)** Could you have written `sync` without `friend` in either class? If yes, what would you need to add to the classes to make it work?
- **(d)** Now extend the code in the next cell. Write a second free function `void report(const Thermostat& t, const Heater& h)` that prints a line like `target=70, current=65, heater on` — reading `target_` and `current_` directly from `t` and `on_` directly from `h` (no getters). You will need to add a second friend declaration in **each** class. Call `report(t, h)` from `main` after each `sync` call to verify.

In [ ]:
////thermostatHeater.cpp
#include<iostream>

class Heater;  // forward declaration so Thermostat can name it

class Thermostat {
private:
    double target_;
    double current_;
public:
    Thermostat(double target, double current) : target_(target), current_(current) {}
    friend void sync(const Thermostat& t, Heater& h);
};

class Heater {
private:
    bool on_;
public:
    Heater() : on_(false) {}
    bool isOn() const { return on_; }
    friend void sync(const Thermostat& t, Heater& h);
};

void sync(const Thermostat& t, Heater& h){
    h.on_ = (t.current_ < t.target_);
}

int main(){
    Thermostat t(70, 65);   // target 70, current 65 — too cold
    Heater h;
    sync(t, h);
    std::cout << "heater on? " << h.isOn() << "\n";   // expect: 1

    Thermostat t2(70, 72);  // target 70, current 72 — warm enough
    Heater h2;
    sync(t2, h2);
    std::cout << "heater on? " << h2.isOn() << "\n";  // expect: 0
}